# AdaptAuth-ML v11 · Ejecución de la validación (PI 13)

Corrida congelada de la validación planificada en el PI 11 + PI 12. Tesista: Ashly Herminia Jasmín Chávez Marón · Asesor: Ing. Mg. Daniel Alfonso Quinto Pazce · UNMSM, Tesis II.

| Paso | Casos de prueba | Tiempo aprox. |
|---|---|---|
| 1 | Código en el commit congelado `5f6f3b1` y versiones exactas de librerías | 1 min |
| 2 | SuspiciousLogin v1 (commit 68cf94b) | segundos |
| 3 | CP01 a CP02: 5 semillas, línea base calibrada a igual FPR, bootstrap por usuario, McNemar, curva recall-FPR | 1 min |
| 4 | CP08, CP11, CP12: requisitos (contingencia, latencias, explicaciones, privacidad) | 2 a 4 min |
| 5 | CP09, CP10: ablación de componentes | 3 a 5 min |
| 6 | CP05: arranque en frío | 5 a 10 min |
| 7 | CP06, CP07: CERT r4.2 con bootstrap por empleado (lee los datos de Drive) | 15 a 25 min |

Los resultados de mayo y junio de SuspiciousLogin son **no confirmatorios**: ese periodo se usó en versiones anteriores.

## Paso 1. Código congelado y entorno

In [ ]:
import os
os.chdir("/content")
!rm -rf adaptauth-ml
!git clone -q https://github.com/AshlyChavez/adaptauth-ml.git
os.chdir("/content/adaptauth-ml")
!git checkout -q 5f6f3b1  # commit congelado de la validación
!pip -q install -r requirements-validacion.txt
!git log --oneline -1
import sys, sklearn, numpy, pandas, scipy, platform
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__, "| scipy", scipy.__version__)

Si scikit-learn no muestra 1.8.0, reinicie la sesión (*Entorno de ejecución → Reiniciar sesión*) y vuelva a ejecutar desde el paso 2.

## Paso 2. SuspiciousLogin v1

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display, Markdown
if not os.path.exists("/content/SL"):
    !git clone -q https://github.com/salomaopena/SuspiciousLoginDataset.git /content/SL
    !git -C /content/SL checkout -q 68cf94b
SL = glob.glob("/content/SL/**/suspicious_logins_public_v1.csv", recursive=True)[0]
print(SL)

## Paso 3. CP01 y CP02: AdaptAuth-ML frente a la línea base calibrada

In [ ]:
!python run_validacion_pi13.py --data "{SL}"
display(pd.read_csv("results_validacion_pi13/cp01_semillas_sl.csv").round(4))
display(pd.read_csv("results_validacion_pi13/o6_recall_fpr_sl.csv").round(4))
display(Image("results_validacion_pi13/figuras/recall_fpr_sl.png", width=700))

## Paso 4. CP08, CP11 y CP12: requisitos

In [ ]:
!python run_requisitos_sl.py --data "{SL}"
display(pd.read_csv("results_requisitos_sl/matriz_cumplimiento_requisitos.csv"))

## Paso 5. CP09 y CP10: ablación de componentes

In [ ]:
!python run_ablacion_componentes.py --data "{SL}" 

## Paso 6. CP05: arranque en frío

In [ ]:
!python run_levantamiento.py --data "{SL}"
display(Image("results_levantamiento/arranque_en_frio_sl.png", width=700))

## Paso 7. CP06 y CP07: CERT r4.2 (datos en Mi unidad → AdaptAuth-ML → CERT_r4.2)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CERT = "/content/drive/MyDrive/AdaptAuth-ML/CERT_r4.2"
!python run_cert_v2.py --dir "{CERT}" --frio
DEST = "/content/drive/MyDrive/AdaptAuth-ML/resultados_pi13"
!mkdir -p "{DEST}"
!cp -r results_validacion_pi13 results_requisitos_sl results_ablacion_componentes results_levantamiento results_cert_v11 "{DEST}/"
print(json.dumps(json.load(open("results_cert_v11/validacion_pi13_cert.json")), indent=1, ensure_ascii=False)[:3000])